#  Week 5, Day 3 — June 17, 2026


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw":BASE_DIR+"/data/raw","processed":BASE_DIR+"/data/processed",
        "metadata":BASE_DIR+"/data/metadata","visualizations":BASE_DIR+"/visualizations"}
print("Environment ready ")
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
df_c = pd.read_csv(DIRS['processed']+'/climate_clean_v1.csv', parse_dates=['Date'])
print(f'climate: {df_c.shape}')

## Step 1: Feature Selection

In [ ]:
FEATURES = ['SST (°C)', 'pH Level', 'Species Observed']
cluster_data = df_c[FEATURES].dropna().copy()

print(f'Feature matrix: {cluster_data.shape[0]} rows × {len(FEATURES)} features')
print()
print('PRE-SCALING RANGES (raw units):')
for col in FEATURES:
    print(f'  {col:<22}: mean={cluster_data[col].mean():.4f}, std={cluster_data[col].std():.4f}, '
          f'min={cluster_data[col].min():.4f}, max={cluster_data[col].max():.4f}')
print()
print('Problem: Species std≈20.45 >> SST std≈1.42 >> pH std≈0.056')
print('Without scaling, Species Observed dominates K-Means distance calculations.')
print('Solution: StandardScaler → mean=0, std=1 for each feature.')

## Step 2: Apply StandardScaler

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_data)
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES)

print('SCALER PARAMETERS (stored for reproducibility):')
for col,mean,std in zip(FEATURES, scaler.mean_, scaler.scale_):
    print(f'  {col:<22}: mean_stored={mean:.4f}, scale_stored={std:.4f}')
print()
print('POST-SCALING VERIFICATION (should all be mean≈0, std≈1):')
print(X_scaled_df.describe().round(4))
print()
print(' All features now on equal footing for distance calculations')

import datetime
scaler_params = {'features':FEATURES,
    'means':[round(float(m),6) for m in scaler.mean_],
    'scales':[round(float(s),6) for s in scaler.scale_],
    'n_samples_fit':int(cluster_data.shape[0])}
with open(DIRS['metadata']+'/kmeans_scaler.json','w') as f: json.dump(scaler_params,f,indent=2)
print('Scaler params saved to kmeans_scaler.json ')

## Step 3: Elbow Method — Confirm k=3

In [ ]:
inertias = {}
for k in range(2,7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X_scaled); inertias[k] = round(km.inertia_,4)

print('ELBOW METHOD — Inertia by k:')
print(f'  {"k":<5} {"Inertia":>12}  {"Δ Inertia":>12}  Note')
print('-'*55)
prev = None
for k,inertia in inertias.items():
    delta = f'{inertia-prev:.4f}' if prev else '—'
    elbow = ' ← ELBOW (k=3 chosen)' if k==3 else ''
    print(f'  {k:<5} {inertia:>12.4f}  {delta:>12}  {elbow}')
    prev = inertia
print()
print('k=3 selected: steepest relative inertia drop (860.65 → 666.95 = Δ193.70)')
print('k=4 drop is smaller (666.95 → 543.37 = Δ123.58) — diminishing returns.')
print('k=3 maps cleanly to project risk categories: Critical / At-Risk / Stable.')

## Step 4: Elbow Curve + Scaled Feature Distribution

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(15,6))

ax1 = axes[0]
ax1.plot(list(inertias.keys()), list(inertias.values()), 'o-', color='steelblue', linewidth=2.5, markersize=9)
ax1.axvline(x=3, color='red', linestyle='--', linewidth=2, label='k=3 chosen')
ax1.fill_between([2,3],[min(inertias.values())-10]*2,[max(inertias.values())+10]*2,alpha=0.08,color='red')
for k,v in inertias.items():
    ax1.annotate(f'{v:.0f}',(k,v),textcoords='offset points',xytext=(0,10),ha='center',fontsize=9)
ax1.set_xlabel('Number of Clusters (k)',fontsize=11)
ax1.set_ylabel('Inertia (Within-Cluster SS)',fontsize=11)
ax1.set_title('Elbow Method — Optimal k=3K-Means Inertia vs Cluster Count',fontweight='bold')
ax1.legend(); ax1.grid(True,alpha=0.3)

ax2 = axes[1]
for col in FEATURES:
    ax2.hist(X_scaled_df[col], bins=30, alpha=0.5, label=col, density=True)
ax2.axvline(0, color='black', linestyle='--', alpha=0.5, label='mean=0')
ax2.set_xlabel('Standardized Value (z-score)',fontsize=11)
ax2.set_ylabel('Density',fontsize=11)
ax2.set_title('Feature Distributions After StandardScaler(All features: mean=0, std=1)',fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True,alpha=0.3)

plt.suptitle('June 17, 2026 — Feature Scaling & Elbow MethodK-Means Setup for Risk Zone Segmentation',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week5_Day3_scaling_elbow.png',dpi=150,bbox_inches='tight')
plt.show(); print('Scaling + Elbow chart saved ')

In [ ]:
scaled_df = cluster_data.copy()
scaled_df[['SST_scaled','pH_scaled','Species_scaled']] = X_scaled
scaled_df.to_csv(DIRS['processed']+'/cluster_features_scaled.csv', index=False)
print(f'Scaled feature matrix saved: cluster_features_scaled.csv ({len(scaled_df)} rows) ✅')
print()
print('K-Means initialized with:')
print('  n_clusters  = 3  (Critical / At-Risk / Stable)')
print('  random_state= 42 (reproducibility)')
print('  n_init      = 10 (10 random restarts — best inertia kept)')
print('  max_iter    = 300')
print()
print('Ready to fit in June 18 notebook.')

## ✅ June 17 Complete
**Elbow confirms k=3. Scaler: SST mean=28.5372, pH mean=8.0499, Species mean=120.472. cluster_features_scaled.csv saved.**

**Next: June 18 — K-Means Execution & Risk Label Assignment**